# Módulo 1 · Etapa 0 — Entorno de trabajo
**TFM · Clasificación de lesiones cutáneas (HAM10000) con CNN y Grad-CAM**
Responsable: **Adrián Arias** · Experimento MLflow de prueba: `sandbox`

**Qué deja hecho este notebook**
1. La carpeta compartida `TFM/` en tu Drive, con su estructura.
2. El repositorio del grupo clonado y las dependencias instaladas.
3. HAM10000 **original** (600×450, sin redimensionar) guardado en Drive como `TFM/data/HAM10000.zip`, en el formato que espera `configs/base.yaml`.
4. La comprobación de que hay 10.015 imágenes, que cada fila de metadatos tiene su imagen, y la huella SHA-256 del CSV (`dataset_sha256`).
5. Las versiones de todas las librerías anotadas en Drive.
6. Un run de prueba en MLflow (DagsHub) que debes ver en la web.

---
**Antes de ejecutar (una sola vez):**
- Icono de la **llave** (barra izquierda) → añade estos secretos y activa el interruptor "Acceso del notebook" en cada uno:

| Secreto | De dónde sale |
|---|---|
| `DAGSHUB_USER` | Tu usuario de DagsHub |
| `DAGSHUB_TOKEN` | DagsHub → *User Settings → Tokens* |
| `KAGGLE_API_TOKEN` | kaggle.com/settings/api → *Generate New Token* → copiar el token |

## 0. Parámetros de la sesión

In [ ]:
AUTOR = "adrian"                          # se guarda como tag en MLflow
OWNER = "aReinoso007"                     # dueño del repositorio en GitHub / DagsHub
REPO  = "tfm-melanoma-cnn-gradcam"
PROJECT_SEED = 42                         # semilla global del proyecto (= split_seed de base.yaml)

## 1. GPU

In [ ]:
# Comprueba qué GPU asignó Colab. En esta etapa no es imprescindible;
# si sale "Sin GPU", puedes seguir igualmente.
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "Sin GPU (entorno CPU)"

## 2. Google Drive y carpeta compartida `TFM/`

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
DRIVE_ROOT = Path("/content/drive/MyDrive/TFM")
for sub in ["data", "checkpoints", "results/m1", "figures/m1"]:
    (DRIVE_ROOT / sub).mkdir(parents=True, exist_ok=True)

print("Estructura en Drive:")
for p in sorted(DRIVE_ROOT.rglob("*")):
    if p.is_dir():
        print("  ", p.relative_to(DRIVE_ROOT.parent))

## 3. Repositorio del grupo y dependencias

In [ ]:
# El repo es público en GitHub, así que se puede clonar sin token.
import os
REPO_DIR = f"/content/{REPO}"
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/{OWNER}/{REPO}.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
%cd {REPO_DIR}
!git log --oneline -1

In [ ]:
# Instala las librerías de requirements.txt (TensorFlow ya viene en Colab) y kagglehub para la descarga.
!pip install -q -r requirements.txt kagglehub

## 4. Secretos

In [ ]:
# Comprueba que los secretos existen SIN imprimir su valor.
from google.colab import userdata

def get_secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None

estado = {}
for name in ["DAGSHUB_USER", "DAGSHUB_TOKEN", "KAGGLE_API_TOKEN"]:
    estado[name] = bool(get_secret(name))
    print(("[+]" if estado[name] else "[!]"), name)

if not all(estado.values()):
    print("\nFalta algún secreto o no tiene activado 'Acceso del notebook' (icono de la llave).")

## 5. Descarga de HAM10000 y copia en Drive
Solo la primera vez (≈ 10–15 min). Si `TFM/data/HAM10000.zip` ya existe, la celda no hace nada.

Se guardan las imágenes **originales**, no redimensionadas: el tamaño final (224, 260 o 300) y el tipo de recorte se deciden más adelante (pipeline y ablación), y así no hay que volver a descargar.

Fuente: Kaggle, `kmader/skin-cancer-mnist-ham10000` (copia de HAM10000; para citar se usa Tschandl et al., 2018). Ese paquete trae las carpetas de imágenes **duplicadas**; la celda se queda con una copia de cada imagen y comprueba que las dos copias pesan lo mismo.

In [ ]:
import glob, shutil, zipfile, time
import pandas as pd

ZIP_DRIVE = DRIVE_ROOT / "data" / "HAM10000.zip"

if ZIP_DRIVE.exists():
    print(f"Ya existe {ZIP_DRIVE} ({ZIP_DRIVE.stat().st_size/1e9:.2f} GB). No se descarga de nuevo.")
else:
    # kagglehub lee el token del secreto KAGGLE_API_TOKEN; se pasa también como variable por si acaso.
    if get_secret("KAGGLE_API_TOKEN"):
        os.environ["KAGGLE_API_TOKEN"] = get_secret("KAGGLE_API_TOKEN")
    import kagglehub
    t0 = time.time()
    root = kagglehub.dataset_download("kmader/skin-cancer-mnist-ham10000")
    print(f"Descargado en {root} ({time.time()-t0:.0f} s)")

    meta_src = glob.glob(f"{root}/**/HAM10000_metadata.csv", recursive=True)[0]

    # Una ruta por image_id; si aparece dos veces, se comprueba que es el mismo archivo.
    imgs, dup_distintas = {}, 0
    for f in sorted(glob.glob(f"{root}/**/*.jpg", recursive=True)):
        iid = Path(f).stem
        if iid in imgs:
            if os.path.getsize(f) != os.path.getsize(imgs[iid]):
                dup_distintas += 1
        else:
            imgs[iid] = f
    print("Imágenes únicas:", len(imgs), "| duplicados con distinto tamaño:", dup_distintas)
    assert dup_distintas == 0, "[!] Hay duplicados distintos."

    # Zip SIN compresión (los JPEG ya están comprimidos): se crea en disco local y luego se copia a Drive.
    tmp = Path("/content/HAM10000.zip")
    with zipfile.ZipFile(tmp, "w", zipfile.ZIP_STORED) as z:
        z.write(meta_src, "HAM10000_metadata.csv")
        for iid, f in imgs.items():
            z.write(f, f"images/{iid}.jpg")
    shutil.copy(tmp, ZIP_DRIVE)
    tmp.unlink()
    print(f"[+] Guardado en Drive: {ZIP_DRIVE} ({ZIP_DRIVE.stat().st_size/1e9:.2f} GB, {time.time()-t0:.0f} s en total)")

## 6. Rutina de cada sesión: copiar el dataset al disco local
Leer miles de imágenes desde Drive es muy lento, así que en **cada sesión** se copia el zip a `/content` y se descomprime allí (≈ 1–2 min). Las rutas salen de `configs/base.yaml`.

In [ ]:
from src.tracking import load_config
cfg = load_config("configs/base.yaml")
LOCAL = Path(cfg["paths"]["local_data_dir"])

if not LOCAL.exists():
    t0 = time.time()
    shutil.copy(cfg["paths"]["dataset_zip"], "/content/HAM10000.zip")
    with zipfile.ZipFile("/content/HAM10000.zip") as z:
        z.extractall(LOCAL)
    os.remove("/content/HAM10000.zip")
    print(f"Descomprimido en {LOCAL} ({time.time()-t0:.0f} s)")
print("Contenido:", sorted(p.name for p in LOCAL.iterdir()))

## 7. Comprobación rápida del dataset

In [ ]:
from src.tracking import file_sha256
IMAGES = Path(cfg["paths"]["images_dir"])
META = Path(cfg["paths"]["metadata_csv"])

meta = pd.read_csv(META)
en_disco = {p.stem for p in IMAGES.glob("*.jpg")}
en_meta = set(meta["image_id"])

checks = {
    "Filas de metadatos = 10.015": len(meta) == 10015,
    "Imágenes en disco = 10.015": len(en_disco) == 10015,
    "Toda fila tiene su imagen": len(en_meta - en_disco) == 0,
    "Ninguna imagen sin fila": len(en_disco - en_meta) == 0,
}
for k, ok in checks.items():
    print(("[+]" if ok else "[!]"), k)

DATASET_SHA256 = file_sha256(META)
print("\nColumnas:", list(meta.columns))
print("Lesiones únicas:", meta["lesion_id"].nunique())
print("dataset_sha256 (huella del CSV de metadatos):", DATASET_SHA256)

In [ ]:
# Criterio de "listo": abrir una imagen cualquiera y ver su diagnóstico.
import matplotlib.pyplot as plt
from PIL import Image

fila = meta.sample(1, random_state=PROJECT_SEED).iloc[0]
img = Image.open(IMAGES / f"{fila.image_id}.jpg")
plt.imshow(img); plt.axis("off")
plt.title(f"{fila.image_id} · dx = {fila.dx} · {img.size[0]}×{img.size[1]} · {img.mode}")
plt.show()
print(fila.to_string())

## 8. Semilla y versiones

In [ ]:
# Fija la semilla del proyecto y anota las versiones exactas (requisito de reproducibilidad).
import sys, json, platform, importlib.metadata as im
import numpy as np, tensorflow as tf
from src.tracking import set_seeds

set_seeds(PROJECT_SEED, deterministic=False)

versiones = {"python": platform.python_version(), "tensorflow": tf.__version__,
             "gpu": [d.name for d in tf.config.list_physical_devices("GPU")]}
for pkg in ["numpy", "pandas", "scikit-learn", "matplotlib", "seaborn",
            "opencv-python-headless", "pillow", "mlflow", "dagshub", "pyyaml"]:
    try:
        versiones[pkg] = im.version(pkg)
    except im.PackageNotFoundError:
        versiones[pkg] = "no instalado"

entorno = {"project_seed": PROJECT_SEED, "dataset_sha256": DATASET_SHA256,
           "fuente": "kaggle:kmader/skin-cancer-mnist-ham10000", "versiones": versiones}
out = DRIVE_ROOT / "results" / "m1" / "entorno.json"
out.write_text(json.dumps(entorno, indent=2, ensure_ascii=False))
print(json.dumps(entorno, indent=2, ensure_ascii=False))
print("\n[+] Guardado en", out)

## 9. Run de prueba en MLflow (DagsHub)

In [ ]:
import mlflow
from src.tracking import setup_mlflow, standard_tags

os.environ["MLFLOW_TRACKING_USERNAME"] = get_secret("DAGSHUB_USER") or ""
os.environ["MLFLOW_TRACKING_PASSWORD"] = get_secret("DAGSHUB_TOKEN") or ""
uri = setup_mlflow(cfg)
mlflow.set_experiment(cfg["mlflow"]["experiments"]["sandbox"])

tags = standard_tags(cfg, modulo="m1", fase="entorno", autor=AUTOR, seed=PROJECT_SEED,
                     dataset_sha256=DATASET_SHA256)
with mlflow.start_run(run_name="m1-prueba-entorno", tags=tags) as run:
    mlflow.log_params({"project_seed": PROJECT_SEED, "tensorflow": tf.__version__})
    mlflow.log_metric("n_imagenes", len(en_disco))
    mlflow.log_dict(entorno, "entorno.json")

print("[+] Run creado:", run.info.run_id)
print("Míralo en:", f"https://dagshub.com/{OWNER}/{REPO}/experiments")

## 10. Resumen

In [ ]:
resumen = {
    "Carpeta TFM/ en Drive": DRIVE_ROOT.exists(),
    "HAM10000.zip en Drive": ZIP_DRIVE.exists(),
    "Comprobaciones del dataset": all(checks.values()),
    "entorno.json guardado": out.exists(),
    "Run de prueba en MLflow": bool(run.info.run_id),
}
for k, ok in resumen.items():
    print(("[+]" if ok else "[!]"), k)